In [ ]:
# 텍스트 파일 읽기 -> 문서 분할 -> 문서 조각들을 context로 작성
#   -> 사용자 질문과 함께 프롬프트에 넣기 -> LLM이 답변

!pip install langchain langchain-openai langchain-community python-dotenv
!pip install langchain-splitters tiktoken

!pip install -U langsmith

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnableLambda   # 함수를 체인으로 묶기 위한 래퍼
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
# Document : 단순한 텍스트 묶음이 아니라 '정보 단위를 표현하기 위한 구조체'이다.
# 각각의 Document는 검색 가능한 하나의 문서 단위(chunk)임
# 여러 Document로 나누는 이유는 1) 검색 성능 향상. 2)서로 다른 출처/메타데이터가 있음

import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY가 없어요")

llm = ChatOpenAI(model="gpt-4o-mini", temperature="0.7")

with open('mytext.txt', 'r', encoding='utf-8') as f:
    text = f.read()
# print(text)

docs = [  # LangChain의 Document 객체로 만들기
    Document(page_content=text, metadata={"source":"mytext.txt"})
]
print(f'문서 갯수:{len(docs)}')
print(f'첫문서 앞 50글자 : {docs[0].page_content[:50]}')

print()
# 문서 분할  : 긴 문서를 작은 청크 단위로 나눔
# chunk_size=300자씩 자르기, chunk_overlap=50 앞뒤 청크가 50자 정도 겹치기(의미 유지가 목적)
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f'생성된 청크 수 : {len(chunks)}')
print(f'첫 청크 앞 50글자 : {chunks[0].page_content[:50]}')

# 검색된 문서들을 하나의 context로 합치는 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Runnable 생성
# RunnableLambda는 일반 파이썬 함수를 LangChain의 체인 앞에서 실행 가능한 단계로 바꿔주는 도구
# 입력값 question을 받기는 하나 그 값을 사용하지 않고 chunks 반환
docs_runnable = RunnableLambda(lambda question:chunks)

# ChatPromptTemplate 생성
prompt = ChatPromptTemplate.from_template(
    """
아래 context를 근거로 질문ㅇ에 한국어로 답해줘.

규칙:
1) context에 있는 내용만 근거로 답해.
2) contex에서 답을 찾을 수 없으면 "주어진 문서로 알수없어요" 라고 답해.
3) 답변은 자연스런 한국 표준말을 사용해.

[컨텍스트]
{context}

[질문]
{question}
    """
)

# LCEL 체인 구성'
chain = (
    {
        # 프롬프트에 매핑할 값
        "context":docs_runnable | RunnableLambda(format_docs),
        "question":RunnablePassthrough()   # 질문 내용 그대로 다음 단계로 전달하는 단수 연결자
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 실행
user_question = "극한 더위에 대해 말해 줘"
response = chain.invoke(user_question)
print(f"질문 : {user_question}")
print(f"답변 : {response}")

# 참고
print("--------------")
prompt_input = {
    "context": format_docs(chunks),
    "question":user_question
}
print(prompt.invoke(prompt_input))
